# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

The scraping of `https://www.scrapethissite.com/pages/forms/` in the last section assumes a hardcoded (fixed) no of pages. Can you improve the code by removing the hardcoded no of pages and instead use the `»` button to determine if there are more pages to scrape? Hint: Use a `while` loop.

In [2]:
import requests
import time
from bs4 import BeautifulSoup

def parse_and_extract_rows(soup: BeautifulSoup):
    """
    Extract table rows from the parsed HTML.

    Args:
        soup: The parsed HTML.

    Returns:
        An iterator of dictionaries with the data from the current page.
    """
    header = soup.find('tr')
    headers = [th.text.strip() for th in header.find_all('th')]
    teams = soup.find_all('tr', 'team')
    for team in teams:
        row_dict = {}
        for header, col in zip(headers, team.find_all('td')):
            row_dict[header] = col.text.strip()
        yield row_dict

Answer:

In [3]:
# Import urljoin so we can safely combine the website base URL
# with the relative next-page link from the » button.
from urllib.parse import urljoin

# Starting page
base_url = "https://www.scrapethissite.com/pages/forms/"

# This variable keeps track of the current page we are scraping
current_url = base_url

# This list will store all scraped team rows from all pages
rows = []

# Keep scraping while there is still a page URL to scrape
while current_url is not None:
    
    # Download the current page
    response = requests.get(current_url)
    
    # If the request failed, stop and show the error
    response.raise_for_status()
    
    # Parse the current page's HTML
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Extract all team rows from the current page
    # parse_and_extract_rows(...) gives us row dictionaries one by one
    rows.extend(parse_and_extract_rows(soup))
    
    # Look for the next-page button, which is shown as »
    next_button = soup.find(
        "a",
        string=lambda text: text is not None and text.strip() == "»"
    )
    
    # If there is no » button, we are on the last page
    if next_button is None:
        current_url = None
    
    # If there is a » button, move to the next page URL
    else:
        next_url = next_button["href"]
        current_url = urljoin(base_url, next_url)
        
        # Be polite to the website; don't spam requests like a desperate Shopee bot
        time.sleep(1)

In [ ]:
# Print the total number of rows extracted from all pages
print(len(rows))